# LLMServeLab on Colab (T4)

Runs the Phase 1 and Phase 2 GPU exit checks: vLLM 0.10.2 serving `Qwen/Qwen2.5-7B-Instruct-AWQ` on one Tesla T4, then `env_check`, `smoke` and `llmserve benchmark` on E01.

Before running: **Runtime → Change runtime type → T4 GPU.** Full notes and traps: [`docs/runbooks/colab.md`](https://github.com/vedantpople4/llmservelab/blob/main/docs/runbooks/colab.md).

Never mix Colab and Kaggle runs in one comparison.

## 1. Environment

In [ ]:
!git clone https://github.com/vedantpople4/llmservelab.git /content/llmservelab
%cd /content/llmservelab
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q uv
!uv sync --locked --extra dev --extra mock --extra gpu

`nvidia-smi` must say `Tesla T4`. Any other GPU is a different SKU: change the runtime type and reconnect.

## 2. vLLM server (§6 flags, identical to Kaggle and `docker/compose.yml`)

In [ ]:
import os
import subprocess
import time
import urllib.request

server = subprocess.Popen(
    [
        "uv",
        "run",
        "vllm",
        "serve",
        "Qwen/Qwen2.5-7B-Instruct-AWQ",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--max-model-len",
        "10240",
        "--gpu-memory-utilization",
        "0.90",
        "--dtype",
        "float16",
        "--no-enable-prefix-caching",
        "--max-num-seqs",
        "256",
        "--disable-log-requests",
    ],
    stdout=open("/content/vllm.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

for _ in range(180):  # first start downloads ~6 GB of weights
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        print("vLLM healthy")
        break
    except Exception:
        time.sleep(5)
else:
    print(open("/content/vllm.log").read()[-4000:])
    raise RuntimeError("vLLM did not become healthy")

In [ ]:
!grep -E 'V1 engine|V0|bfloat16|float16' /content/vllm.log | head

## 3. Phase 1 exit check: env check and 100-request smoke

In [ ]:
!uv run python -m llmserve.runner.env_check configs/baseline/e01.yaml
!uv run python scripts/smoke.py configs/baseline/e01.yaml

## 4. Phase 2 exit check (GPU half): full benchmark

In [ ]:
!uv run llmserve benchmark configs/baseline/e01.yaml

## 5. Save results to Drive (after every benchmark; `/content` is wiped on disconnect)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/llmservelab
!cp -r results /content/drive/MyDrive/llmservelab/
!cp /content/vllm.log /content/drive/MyDrive/llmservelab/vllm-$(date +%Y%m%dT%H%M%S).log

When done measuring: **Runtime → Disconnect and delete runtime**, so the free GPU quota stops draining.